# The Student Productivity Equation

In this notebook we take the key findings from our EDA on 20,000 student records and shape them into a story. The goal is to understand what actually drives student productivity — and prepare the ground for the dashboard.

From the EDA we already know that `productivity_score` correlates well with student behavior, while `final_grade` doesn't really respond to anything (probably a quirk of the synthetic data). So here we focus on **productivity** and **focus** as our main outcomes.

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.options.mode.copy_on_write = True

TEMPLATE = "plotly_white"

df = pd.read_csv("data/student_productivity_distraction_dataset_20000.csv")

df["total_distraction_hours"] = (
    df["phone_usage_hours"] + df["social_media_hours"] + df["youtube_hours"] + df["gaming_hours"]
)
df["good_sleep"] = np.where(df["sleep_hours"] >= 7, "Sleep >= 7h", "Sleep < 7h")
df["exercises_regularly"] = np.where(df["exercise_minutes"] >= 60, "Exercise >= 60min", "Exercise < 60min")
df["healthy_habits_score"] = (df["sleep_hours"] >= 7).astype(int) + (df["exercise_minutes"] >= 60).astype(int)
df["distraction_level"] = pd.cut(
    df["total_distraction_hours"],
    bins=[0, 5, 10, 15, df["total_distraction_hours"].max()],
    labels=["Low (0-5h)", "Medium (5-10h)", "High (10-15h)", "Very High (15h+)"],
    include_lowest=True,
)
df["productivity_tier"] = pd.qcut(df["productivity_score"], q=4, labels=["Low", "Mid-Low", "Mid-High", "High"])

print(f"Dataset: {df.shape[0]:,} students, {df.shape[1]} features")
df[
    ["study_hours_per_day", "total_distraction_hours", "sleep_hours", "productivity_score", "focus_score"]
].describe().round(2)

Dataset: 20,000 students, 24 features


/tmp/ipykernel_27348/4095398597.py:7: Pandas4Warning: The 'mode.copy_on_write' option is deprecated. Copy-on-Write can no longer be disabled (it is always enabled with pandas >= 3.0), and setting the option has no impact. This option will be removed in pandas 4.0.
  pd.options.mode.copy_on_write = True


,study_hours_per_day,total_distraction_hours,sleep_hours,productivity_score,focus_score
count,20000.00,20000.00,20000.00,20000.00,20000.00
mean,5.25,16.23,6.52,50.18,64.44
std,2.74,4.73,2.03,16.09,20.18
min,0.50,1.78,3.00,0.00,30.00
25%,2.90,12.87,4.77,38.70,47.00
50%,5.25,16.26,6.51,50.24,65.00
75%,7.64,19.61,8.31,61.78,82.00
max,10.00,31.20,10.00,100.00,99.00


---
## 1. The Time Budget

Before looking at outcomes, let's see where the hours actually go.

In [2]:
time_categories = {
    "Study": df["study_hours_per_day"].mean(),
    "Sleep": df["sleep_hours"].mean(),
    "Phone (non-social)": (df["phone_usage_hours"] - df["social_media_hours"]).clip(lower=0).mean(),
    "Social Media": df["social_media_hours"].mean(),
    "YouTube": df["youtube_hours"].mean(),
    "Gaming": df["gaming_hours"].mean(),
}

time_df = pd.DataFrame(
    {
        "Activity": list(time_categories.keys()),
        "Hours": list(time_categories.values()),
    }
)
time_df["Category"] = ["Productive", "Rest", "Distraction", "Distraction", "Distraction", "Distraction"]

fig = px.sunburst(
    time_df,
    path=["Category", "Activity"],
    values="Hours",
    color="Category",
    color_discrete_map={"Productive": "#2a9d8f", "Rest": "#264653", "Distraction": "#e76f51"},
    template=TEMPLATE,
    title="<b>Where Does the Average Student's Day Go?</b><br><sup>Mean daily hours across 20,000 students</sup>",
)
fig.update_traces(
    textinfo="label+value+percent parent", texttemplate="%{label}<br>%{value:.1f}h (%{percentParent:.0%})"
)
fig.update_layout(width=650, height=550, margin={"t": 80, "b": 20, "l": 20, "r": 20})
fig.show()

In [3]:
fig = go.Figure()
fig.add_trace(
    go.Histogram(
        x=df["study_hours_per_day"],
        name="Study Hours",
        marker_color="#2a9d8f",
        opacity=0.7,
        nbinsx=40,
    )
)
fig.add_trace(
    go.Histogram(
        x=df["total_distraction_hours"],
        name="Total Distraction Hours",
        marker_color="#e76f51",
        opacity=0.7,
        nbinsx=40,
    )
)
fig.update_layout(
    barmode="overlay",
    template=TEMPLATE,
    title="<b>Study vs. Distraction: Daily Hour Distributions</b><br><sup>Overlaid histograms for all students</sup>",
    xaxis_title="Hours per Day",
    yaxis_title="Number of Students",
    legend={"yanchor": "top", "y": 0.95, "xanchor": "right", "x": 0.95},
    width=850,
    height=450,
)
fig.add_vline(
    x=df["study_hours_per_day"].mean(),
    line_dash="dash",
    line_color="#2a9d8f",
    annotation_text=f"Mean study: {df['study_hours_per_day'].mean():.1f}h",
    annotation_position="top left",
)
fig.add_vline(
    x=df["total_distraction_hours"].mean(),
    line_dash="dash",
    line_color="#e76f51",
    annotation_text=f"Mean distraction: {df['total_distraction_hours'].mean():.1f}h",
    annotation_position="top right",
)
fig.show()

Students spend roughly **twice as much time** on distractions as on studying. The distraction distribution is also wider — some students rack up extreme screen time.

---
## 2. The Distraction Tax

So what does all that screen time actually cost?

In [4]:
sample = df.sample(n=3000, random_state=42)

fig = px.scatter(
    sample,
    x="total_distraction_hours",
    y="productivity_score",
    color="stress_level",
    color_continuous_scale="RdYlGn_r",
    trendline="ols",
    opacity=0.5,
    template=TEMPLATE,
    labels={
        "total_distraction_hours": "Total Distraction Hours / Day",
        "productivity_score": "Productivity Score",
        "stress_level": "Stress Level",
    },
    title="<b>More Distraction, Less Productivity</b><br><sup>OLS trendline with stress-level coloring (3,000 sample)</sup>",
)
fig.update_layout(width=850, height=500)
fig.show()

results = px.get_trendline_results(fig)
slope = results.iloc[0]["px_fit_results"].params[1]
print(f"OLS slope: {slope:.2f} productivity points per additional distraction hour")

OLS slope: -0.81 productivity points per additional distraction hour


In [5]:
distraction_long = df.melt(
    id_vars=["student_id", "distraction_level"],
    value_vars=["phone_usage_hours", "social_media_hours", "youtube_hours", "gaming_hours"],
    var_name="distraction_type",
    value_name="hours",
)
distraction_long["distraction_type"] = (
    distraction_long["distraction_type"]
    .str.replace("_hours", "")
    .str.replace("_usage", "")
    .str.replace("_", " ")
    .str.title()
)

fig = px.box(
    df,
    x="distraction_level",
    y="productivity_score",
    color="distraction_level",
    color_discrete_sequence=["#2a9d8f", "#e9c46a", "#f4a261", "#e76f51"],
    template=TEMPLATE,
    title="<b>Productivity by Distraction Level</b><br><sup>Distribution shifts downward as distractions increase</sup>",
    labels={"distraction_level": "Distraction Level", "productivity_score": "Productivity Score"},
    category_orders={"distraction_level": ["Low (0-5h)", "Medium (5-10h)", "High (10-15h)", "Very High (15h+)"]},
)
fig.update_layout(width=850, height=480, showlegend=False)
fig.show()

In [17]:
phone_labels = ["0 - 2", "2 - 4", "4 - 6", "6 - 8", "8 - 10", "10 - 12"]
social_labels = ["0 - 2", "2 - 4", "4 - 6", "6 - 8"]

df["phone_bin"] = pd.cut(
    df["phone_usage_hours"], bins=[0, 2, 4, 6, 8, 10, 12], labels=phone_labels, include_lowest=True
)
df["social_bin"] = pd.cut(df["social_media_hours"], bins=[0, 2, 4, 6, 8], labels=social_labels, include_lowest=True)

heatmap_data = df.pivot_table(
    index="phone_bin",
    columns="social_bin",
    values="productivity_score",
    aggfunc="mean",
    observed=False,
)

z_vals = heatmap_data.to_numpy().round(1)
x_labels = [str(c) for c in heatmap_data.columns]
y_labels = [str(r) for r in heatmap_data.index]

fig = go.Figure(
    data=go.Heatmap(
        z=z_vals,
        x=x_labels,
        y=y_labels,
        colorscale="RdYlGn",
        text=z_vals,
        texttemplate="%{text:.1f}",
        textfont={"size": 13},
        colorbar={"title": "Mean<br>Productivity"},
    )
)
fig.update_layout(
    template=TEMPLATE,
    width=800,
    height=500,
    title="<b>Distraction Interaction: Phone * Social Media</b><br><sup>Mean productivity by usage combination</sup>",
    xaxis_title="Social Media (h/day)",
    yaxis_title="Phone Usage (h/day)",
    yaxis={"autorange": "reversed"},
)
fig.show()

df = df.drop(columns=["phone_bin", "social_bin"])

The relationship is roughly linear — more distraction, less productivity. The heatmap also shows that high phone **and** high social media together is worse than either alone.

---
## 3. The Study–Productivity Pipeline

Now the flip side — how study effort translates into productivity.

In [7]:
sample = df.sample(n=3000, random_state=42)

fig = px.scatter(
    sample,
    x="study_hours_per_day",
    y="productivity_score",
    color="focus_score",
    color_continuous_scale="Viridis",
    trendline="ols",
    trendline_color_override="#e76f51",
    opacity=0.45,
    template=TEMPLATE,
    labels={
        "study_hours_per_day": "Study Hours / Day",
        "productivity_score": "Productivity Score",
        "focus_score": "Focus Score",
    },
    title="<b>Study Hours Drive Productivity</b><br><sup>Colored by focus score, with OLS trendline</sup>",
)
fig.update_layout(width=900, height=530)
fig.show()

r = df["study_hours_per_day"].corr(df["productivity_score"])
print(f"Pearson r (study hours vs productivity): {r:.3f}")

Pearson r (study hours vs productivity): 0.733


In [8]:
academic_vars = ["study_hours_per_day", "focus_score", "assignments_completed", "attendance_percentage"]
corrs = df[academic_vars].corrwith(df["productivity_score"]).sort_values()

fig = px.bar(
    x=corrs.values,
    y=[v.replace("_", " ").title() for v in corrs.index],
    orientation="h",
    template=TEMPLATE,
    title="<b>What Drives Productivity?</b><br><sup>Pearson r with productivity score</sup>",
    labels={"x": "Correlation (r)", "y": ""},
    text=[f"{v:.2f}" for v in corrs.values],
)
fig.update_traces(marker_color=["#e76f51" if v < 0.2 else "#2a9d8f" for v in corrs.values])
fig.update_layout(width=750, height=350, yaxis={"tickfont": {"size": 13}})
fig.show()

Study hours are the strongest predictor (r ≈ 0.73). In the parallel coordinates you can see that high-productivity students (dark lines) sit at the top on every axis. Focus seems to act as a mediator — it's the bridge between study time and the actual outcome.

---
## 4. Lifestyle as Amplifier

Sleep, exercise, and stress don't replace study effort, but they can boost or undermine it.

In [9]:
fig = px.box(
    df,
    x="good_sleep",
    y="productivity_score",
    color="exercises_regularly",
    color_discrete_map={"Exercise >= 60min": "#2a9d8f", "Exercise < 60min": "#e76f51"},
    template=TEMPLATE,
    labels={
        "good_sleep": "Sleep Category",
        "productivity_score": "Productivity Score",
        "exercises_regularly": "Exercise Habit",
    },
    title="<b>Sleep + Exercise: Compound Lifestyle Effects</b><br><sup>Productivity distributions by sleep and exercise habits</sup>",
    category_orders={
        "good_sleep": ["Sleep < 7h", "Sleep >= 7h"],
        "exercises_regularly": ["Exercise < 60min", "Exercise >= 60min"],
    },
)
fig.update_layout(
    width=800,
    height=500,
    legend={"title": "Exercise Habit", "yanchor": "top", "y": 0.95, "xanchor": "right", "x": 0.98},
    boxmode="group",
)
fig.show()

for sleep_label in ["Sleep < 7h", "Sleep >= 7h"]:
    for ex_label in ["Exercise < 60min", "Exercise >= 60min"]:
        mask = (df["good_sleep"] == sleep_label) & (df["exercises_regularly"] == ex_label)
        mean_p = df.loc[mask, "productivity_score"].mean()
        print(f"  {sleep_label} + {ex_label}: mean productivity = {mean_p:.1f}")

  Sleep < 7h + Exercise < 60min: mean productivity = 46.1
  Sleep < 7h + Exercise >= 60min: mean productivity = 45.9
  Sleep >= 7h + Exercise < 60min: mean productivity = 55.4
  Sleep >= 7h + Exercise >= 60min: mean productivity = 56.0


In [10]:
stress_agg = df.groupby("stress_level", as_index=False).agg(
    mean_focus=("focus_score", "mean"),
    mean_productivity=("productivity_score", "mean"),
    count=("student_id", "count"),
)

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(
    go.Scatter(
        x=stress_agg["stress_level"],
        y=stress_agg["mean_productivity"],
        mode="lines+markers",
        name="Productivity",
        line={"color": "#2a9d8f", "width": 3},
        marker={"size": 8},
    ),
    secondary_y=False,
)
fig.add_trace(
    go.Scatter(
        x=stress_agg["stress_level"],
        y=stress_agg["mean_focus"],
        mode="lines+markers",
        name="Focus Score",
        line={"color": "#e9c46a", "width": 3, "dash": "dot"},
        marker={"size": 8},
    ),
    secondary_y=True,
)
fig.update_layout(
    template=TEMPLATE,
    width=850,
    height=450,
    title="<b>Stress Erodes Both Focus and Productivity</b><br><sup>Mean scores by self-reported stress level (1–10)</sup>",
    xaxis_title="Stress Level",
    legend={"yanchor": "top", "y": 0.95, "xanchor": "right", "x": 0.95},
)
fig.update_yaxes(title_text="Mean Productivity Score", secondary_y=False)
fig.update_yaxes(title_text="Mean Focus Score", secondary_y=True)
fig.show()

In [11]:
lifestyle_vars = ["sleep_hours", "exercise_minutes", "stress_level", "coffee_intake_mg"]
corrs = df[lifestyle_vars].corrwith(df["productivity_score"]).sort_values()

labels = [
    v.replace("_", " ").replace("mg", "(mg)").replace("minutes", "(min)").replace("hours", "(h)").title()
    for v in corrs.index
]
colors = ["#e76f51" if v < 0 else "#2a9d8f" for v in corrs.values]

fig = px.bar(
    x=corrs.values,
    y=labels,
    orientation="h",
    template=TEMPLATE,
    title="<b>Lifestyle vs Productivity</b><br><sup>Pearson r with productivity score</sup>",
    labels={"x": "Correlation (r)", "y": ""},
    text=[f"{v:+.2f}" for v in corrs.values],
)
fig.update_traces(marker_color=colors)
fig.update_layout(width=700, height=320, yaxis={"tickfont": {"size": 13}})
fig.show()

Good sleep + exercise together give the best productivity bump. Stress is a clear drag on both focus and productivity. Coffee doesn't seem to help or hurt — students probably just drink more of it when they're already struggling.

---
## 5. Student Profiles

Let's segment students by distraction level and productivity tier to see who's who.

In [12]:
sunburst_agg = (
    df.groupby(["distraction_level", "productivity_tier"], observed=True)
    .agg(count=("student_id", "count"), mean_focus=("focus_score", "mean"))
    .reset_index()
)

fig = px.sunburst(
    sunburst_agg,
    path=["distraction_level", "productivity_tier"],
    values="count",
    color="mean_focus",
    color_continuous_scale="Viridis",
    template=TEMPLATE,
    title="<b>Student Profiles: Distraction Level x Productivity Tier</b><br><sup>Size = student count, color = mean focus score</sup>",
    labels={"mean_focus": "Mean Focus"},
)
fig.update_traces(textinfo="label+value", texttemplate="%{label}<br>n=%{value:,}")
fig.update_layout(width=750, height=620, margin={"t": 80, "b": 20, "l": 20, "r": 20})
fig.show()

---
## Summary

Three things drive student productivity:

| Factor | Effect |
|--------|--------|
| Study hours | Strong positive (r ≈ 0.73) |
| Distractions (phone, social media, etc.) | Negative — roughly linear |
| Lifestyle (sleep, exercise, stress) | Amplifies or dampens everything else |

For the dashboard, the plan is to turn these sections into interactive panels — filters by distraction type, brushable parallel coordinates, and a student segmentation view.